In [35]:
from google.colab import files
from Bio.PDB import PDBParser
import numpy as np
import itertools
uploaded = files.upload()
#need to choose 1YCR.pcb file from computer when prompted

Saving 1YCR.pdb to 1YCR (2).pdb


In [36]:
!pip install biopython numpy

In [37]:
# Angles in degrees, top 3 most common rotamers per AA

#rotamers defined by pocket residues and p53 peptides, angles found in (https://onlinelibrary.wiley.com/doi/full/10.1002/1097-0134%2820000815%2940%3A3%3C389%3A%3AAID-PROT50%3E3.0.CO%3B2-2?utm_sq=grq7sdtfar)
ROTAMER_LIB = {
    'LEU': [(-65, 175), (-177, 65), (-85, 65)],
    'VAL': [(-175, None), (-60, None), (63, None)],
    'ILE': [(-65, 170), (-57, 60), (62, 170)],
    'MET': [(-65, -65), (-67, 180), (-67, 180)],
    'PHE': [(-65, -85), (-177, 80), (62, 90)],
    'TYR': [(-65, -85), (-177, 80), (62, 90)],
    'HIS': [(-65, -70), (-177, 60), (-65, 80)],
    'GLY': [(None, None)],
    'GLN': [(-67, 180), (-177, 180), (-65, -65)],
    'THR': [(62, None), (-65, None), (-175, None)],
    'SER': [(-62, None), (-65, None), (-177, None)],
    'TRP': [(-65, 95), (-177, 90), (-177, -105)],

}

In [38]:
def get_sidechain_centroid(residue, chi1_deg):
    try:
        ca = residue['CA'].get_vector()
        cb = residue['CB'].get_vector()
    except KeyError:
        ca = residue['CA'].get_vector()
        return np.array([ca[0], ca[1], ca[2]])

    # Direction from CA to CB
    ca_arr = np.array([ca[0], ca[1], ca[2]])
    cb_arr = np.array([cb[0], cb[1], cb[2]])
    direction = cb_arr - ca_arr
    direction = direction / np.linalg.norm(direction)

    # Rotate direction by chi1 around CA-CB axis
    if chi1_deg is not None:
        chi1_rad = np.radians(chi1_deg)
        perp = np.array([-direction[1], direction[0], 0])
        if np.linalg.norm(perp) < 1e-6:
            perp = np.array([0, -direction[2], direction[1]])
        perp = perp / np.linalg.norm(perp)
        rotated = (np.cos(chi1_rad) * direction +
                   np.sin(chi1_rad) * perp)
        centroid = cb_arr + 2.5 * rotated
    else:
        centroid = cb_arr + 1.5 * direction

    return centroid


def get_pocket_centroids(pocket_residues):
    centroids = []
    for res in pocket_residues:
        try:
            cb = res['CB'].get_vector()
            centroids.append(np.array([cb[0], cb[1], cb[2]]))
        except KeyError:
            ca = res['CA'].get_vector()
            centroids.append(np.array([ca[0], ca[1], ca[2]]))
    return np.array(centroids)

In [39]:
HYDROPHOBIC = {'PHE', 'TRP', 'LEU', 'ILE', 'VAL', 'MET', 'TYR'}

def vdw_energy(dist, r_min=3.5, epsilon=1):
    if dist < 1.0:
        return 100.0  # cap
    ratio = r_min / dist
    lj = epsilon*(ratio**12 - 2 * ratio**6)
    return min(lj, 100.0)  # soft cap on repulsion


def hydrophobic_score(aa_name, sidechain_pos, pocket_centroids, cutoff=5.0):
    # Count pocket contacts within cutoff
    dists = np.linalg.norm(pocket_centroids - sidechain_pos, axis=1)
    n_contacts = np.sum(dists < cutoff)

    if aa_name in HYDROPHOBIC:
        return -0.5 * n_contacts   # reward burial
    else:
        return 0.3 * n_contacts   # penalize polar burial


def self_energy(aa_name, sidechain_pos, pocket_residues, pocket_centroids):
    total = 0.0

    # vdW with each pocket residue
    for pc in pocket_centroids:
        dist = np.linalg.norm(sidechain_pos - pc)
        total += vdw_energy(dist)

    # hydrophobic burial
    total += hydrophobic_score(aa_name, sidechain_pos, pocket_centroids)

    return total


def pair_energy(pos1, pos2):
    dist = np.linalg.norm(pos1 - pos2)
    return vdw_energy(dist)

In [40]:
def build_energy_matrices(peptide_residues, pocket_residues, design_positions):
    # Pre-compute all self and pair energies for every rotamer at every position.

    pocket_centroids = get_pocket_centroids(pocket_residues)

    all_rotamers = {}

    for pos_idx in design_positions:
        res = peptide_residues[pos_idx]
        rotamers_at_pos = []

        for aa, chi_list in ROTAMER_LIB.items():
            for chi1, chi2 in chi_list:
                centroid = get_sidechain_centroid(res, chi1)
                se = self_energy(aa, centroid, pocket_residues, pocket_centroids)
                rotamers_at_pos.append({
                    'aa': aa,
                    'chi1': chi1,
                    'chi2': chi2,
                    'centroid': centroid,
                    'self_e': se
                })

        all_rotamers[pos_idx] = rotamers_at_pos
        print(f"Position {pos_idx} ({res.resname}): {len(rotamers_at_pos)} rotamers")

    # Pre-compute pairwise energies between design positions
    pair_energies = {}
    for i in design_positions:
        for j in design_positions:
            if j <= i:
                continue
            pair_energies[(i,j)] = np.zeros(
                (len(all_rotamers[i]), len(all_rotamers[j]))
            )
            for ri, rot_i in enumerate(all_rotamers[i]):
                for rj, rot_j in enumerate(all_rotamers[j]):
                    pair_energies[(i,j)][ri][rj] = pair_energy(
                        rot_i['centroid'], rot_j['centroid']
                    )

    return all_rotamers, pair_energies

In [41]:
def run_dee(all_rotamers, pair_energies, design_positions):
    alive = {pos: list(range(len(all_rotamers[pos]))) for pos in design_positions}

    def get_pair_e(i, r_idx, j, s_idx):
        if (i, j) in pair_energies:
            return pair_energies[(i,j)][r_idx][s_idx]
        else:
            return pair_energies[(j,i)][s_idx][r_idx]

    total_eliminated = 0
    iteration = 0

    while True:
        eliminated_this_round = 0
        iteration += 1

        for i in design_positions:
            to_eliminate = []

            for r_idx in alive[i]:
                r_self = all_rotamers[i][r_idx]['self_e']

                for t_idx in alive[i]:
                    if t_idx == r_idx:
                        continue

                    t_self = all_rotamers[i][t_idx]['self_e']

                    dee_sum = r_self - t_self

                    for j in design_positions:
                        if j == i:
                            continue

                        alive_j = alive[j]
                        if not alive_j:
                            continue

                        min_r = min(get_pair_e(i, r_idx, j, s) for s in alive_j)
                        max_t = max(get_pair_e(i, t_idx, j, s) for s in alive_j)
                        dee_sum += min_r - max_t

                    if dee_sum > 0:
                        to_eliminate.append(r_idx)
                        break

            for r_idx in to_eliminate:
                if r_idx in alive[i]:
                    alive[i].remove(r_idx)
                    eliminated_this_round += 1

        total_eliminated += eliminated_this_round
        print(f"DEE iteration {iteration}: eliminated {eliminated_this_round} rotamers")

        if eliminated_this_round == 0:
            break

    print(f"\nDEE complete. Total eliminated: {total_eliminated}")
    for pos in design_positions:
        print(f"  Position {pos}: {len(alive[pos])} rotamers remaining")

    return alive

In [42]:
def exhaustive_search(all_rotamers, pair_energies, design_positions, alive):
    def get_pair_e(i, r_idx, j, s_idx):
        if (i, j) in pair_energies:
            return pair_energies[(i,j)][r_idx][s_idx]
        else:
            return pair_energies[(j,i)][s_idx][r_idx]

    alive_lists = [alive[pos] for pos in design_positions]

    total_combos = 1
    for lst in alive_lists:
        total_combos *= len(lst)
    print(f"\nExhaustive search over {total_combos} combinations...")

    best_energy = float('inf')
    best_combo = None

    for combo in itertools.product(*alive_lists):
        total_e = 0.0

        # Self energies
        for k, pos in enumerate(design_positions):
            total_e += all_rotamers[pos][combo[k]]['self_e']

        # Pair energies
        for k1, pos1 in enumerate(design_positions):
            for k2, pos2 in enumerate(design_positions):
                if pos2 <= pos1:
                    continue
                total_e += get_pair_e(pos1, combo[k1], pos2, combo[k2])

        if total_e < best_energy:
            best_energy = total_e
            best_combo = combo

    return best_combo, best_energy


def report_result(all_rotamers, design_positions, best_combo, best_energy):
    print("\n=== DESIGNED PEPTIDE ===")
    for k, pos in enumerate(design_positions):
        r_idx = best_combo[k]
        rot = all_rotamers[pos][r_idx]
        print(f"  Position {pos}: {rot['aa']}  (chi1={rot['chi1']}, chi2={rot['chi2']})  self_e={rot['self_e']:.2f}")
    print(f"  Total energy: {best_energy:.2f}")

In [43]:
def energy_breakdown(all_rotamers, peptide_residues, pocket_residues,
                     design_positions, best_combo):
    pocket_centroids = get_pocket_centroids(pocket_residues)

    print("\n=== PER POSITION ENERGY BREAKDOWN ===")
    print(f"{'Position':<10} {'Native AA':<12} {'Native E':<12} {'Designed AA':<14} {'Designed E':<12} {'Improvement'}")
    print("-" * 75)

    total_native = 0
    total_designed = 0

    for k, pos in enumerate(design_positions):
        # Designed
        r_idx = best_combo[k]
        rot = all_rotamers[pos][r_idx]
        designed_e = rot['self_e']

        # Native
        res = peptide_residues[pos]
        try:
            cb = res['CB'].get_vector()
            pos_arr = np.array([cb[0], cb[1], cb[2]])
        except KeyError:
            ca = res['CA'].get_vector()
            pos_arr = np.array([ca[0], ca[1], ca[2]])
        native_e = self_energy(res.resname, pos_arr, pocket_residues, pocket_centroids)

        improvement = native_e - designed_e
        total_native += native_e
        total_designed += designed_e

        print(f"{pos:<10} {res.resname:<12} {native_e:<12.2f} {rot['aa']:<14} {designed_e:<12.2f} {improvement:.2f}")

    print("-" * 75)
    print(f"{'Total':<10} {'':<12} {total_native:<12.2f} {'':<14} {total_designed:<12.2f} {total_native-total_designed:.2f}")

In [44]:
def main():
    # Load structure
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('1YCR', '1YCR.pdb')
    model = structure[0]
    mdm2 = model['A']
    p53  = model['B']

    # Identify pocket
    p53_atoms = list(p53.get_atoms())
    pocket_residues = []
    for res in mdm2:
        if res.id[0] != ' ':
            continue
        for atom in res:
            for p53_atom in p53_atoms:
                if atom - p53_atom < 8.0:
                    pocket_residues.append(res)
                    break
            else:
                continue
            break
    print(f"Pocket: {len(pocket_residues)} residues")

    # Extract peptide residues
    peptide_residues = [r for r in p53 if r.id[0] == ' ']
    print(f"Peptide: {len(peptide_residues)} residues")
    for i, r in enumerate(peptide_residues):
        print(f"  [{i}] {r.resname} {r.id[1]}")

    # Choose design positions
    design_positions = [1, 3, 5]

    # Build energy matrices
    all_rotamers, pair_energies = build_energy_matrices(
        peptide_residues, pocket_residues, design_positions
    )

    # Run DEE
    alive = run_dee(all_rotamers, pair_energies, design_positions)

    # Exhaustive search
    best_combo, best_energy = exhaustive_search(
        all_rotamers, pair_energies, design_positions, alive
    )
    report_result(all_rotamers, design_positions, best_combo, best_energy)
    energy_breakdown(all_rotamers, peptide_residues, pocket_residues,
                 design_positions, best_combo)
main()

Pocket: 47 residues
Peptide: 13 residues
  [0] GLU 17
  [1] THR 18
  [2] PHE 19
  [3] SER 20
  [4] ASP 21
  [5] LEU 22
  [6] TRP 23
  [7] LYS 24
  [8] LEU 25
  [9] LEU 26
  [10] PRO 27
  [11] GLU 28
  [12] ASN 29
Position 1 (THR): 34 rotamers
Position 3 (SER): 34 rotamers
Position 5 (LEU): 34 rotamers
DEE iteration 1: eliminated 81 rotamers
DEE iteration 2: eliminated 13 rotamers
DEE iteration 3: eliminated 0 rotamers

DEE complete. Total eliminated: 94
  Position 1: 5 rotamers remaining
  Position 3: 2 rotamers remaining
  Position 5: 1 rotamers remaining

Exhaustive search over 10 combinations...

=== DESIGNED PEPTIDE ===
  Position 1: LEU  (chi1=-177, chi2=65)  self_e=-1.46
  Position 3: VAL  (chi1=-175, chi2=None)  self_e=-0.15
  Position 5: VAL  (chi1=63, chi2=None)  self_e=-1.88
  Total energy: -4.00

=== PER POSITION ENERGY BREAKDOWN ===
Position   Native AA    Native E     Designed AA    Designed E   Improvement
------------------------------------------------------------------

In [45]:
!pip install pyrosetta-installer
import pyrosetta_installer
pyrosetta_installer.install_pyrosetta()

PyRosetta install detected, doing nothing...


In [46]:
import pyrosetta
pyrosetta.init()

from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.core.scoring import get_score_function
from pyrosetta.toolbox import mutate_residue
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.protocols.minimization_packing import MinMover

# Load structure
pose = pose_from_pdb('1YCR.pdb')
sfxn = get_score_function(True)

def get_rosetta_resnum(pose, chain, pdb_resnum):
    return pose.pdb_info().pdb2pose(chain, pdb_resnum)

# Relax side chains only
def relax_pose(pose, sfxn):
    movemap = MoveMap()
    movemap.set_bb(False)
    movemap.set_chi(True)
    min_mover = MinMover()
    min_mover.movemap(movemap)
    min_mover.score_function(sfxn)
    min_mover.apply(pose)
    return pose

pos_list = [('B', 18), ('B', 20), ('B', 22)]

# Score native
native_pose = pose.clone()
native_pose = relax_pose(native_pose, sfxn)
native_score = sfxn(native_pose)

designed_pose = pose.clone()
mutations = {18: 'L', 20: 'V', 22: 'V'}
for pdb_num, aa in mutations.items():
    rosetta_num = get_rosetta_resnum(designed_pose, 'B', pdb_num)
    mutate_residue(designed_pose, rosetta_num, aa)
designed_pose = relax_pose(designed_pose, sfxn)
designed_score = sfxn(designed_pose)

# Per residue breakdown
native_e = native_pose.energies()
designed_e = designed_pose.energies()

print("\n=== PYROSETTA COMPARISON (after relaxation) ===")
print(f"{'PDB Res':<10} {'Native AA':<12} {'Native E':<12} {'Designed AA':<14} {'Designed E'}")
print("-" * 60)

native_aas  = {18: 'THR', 20: 'SER', 22: 'LEU'}
designed_aas = {18: 'LEU', 20: 'VAL', 22: 'VAL'}

for pdb_num in mutations.keys():
    rn = get_rosetta_resnum(native_pose, 'B', pdb_num)
    ne = native_e.residue_total_energy(rn)
    de = designed_e.residue_total_energy(rn)
    print(f"{pdb_num:<10} {native_aas[pdb_num]:<12} {ne:<12.2f} {designed_aas[pdb_num]:<14} {de:.2f}")

print("-" * 60)
print(f"{'Total':<10} {'':<12} {native_score:<12.2f} {'':<14} {designed_score:.2f}")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu 2026.15+release.b166af21f0776040bd54a021d8c7fd264cd99bfa 2026-04-10T14:11:34] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python312.ubuntu r427 2026.15+release.b166af21f0 b1